In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pandas openpyxl

In [4]:
!pip install transformers sentencepiece accelerate

In [5]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/db.py
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).resolve().parents[2] / "database" / "app.db"

def get_conn() -> sqlite3.Connection:
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    return sqlite3.connect(DB_PATH)

def column_exists(conn, table_name, column_name):
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    return any(row[1] == column_name for row in rows)

def migrate() -> None:
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS endpoints (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            url TEXT NOT NULL,
            enabled INTEGER NOT NULL DEFAULT 1,
            expected_latency_ms INTEGER NOT NULL DEFAULT 500
        )
        """)

        conn.execute("""
        CREATE TABLE IF NOT EXISTS check_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            endpoint_id INTEGER NOT NULL,
            ts TEXT NOT NULL,
            latency_ms INTEGER,
            status TEXT NOT NULL,
            http_status INTEGER,
            error_type TEXT,
            error_message TEXT,
            FOREIGN KEY(endpoint_id) REFERENCES endpoints(id)
        )
        """)

        if not column_exists(conn, "check_results", "llm_label"):
            conn.execute("ALTER TABLE check_results ADD COLUMN llm_label TEXT")

        if not column_exists(conn, "check_results", "llm_analysis TEXT"):
            try:
                conn.execute("ALTER TABLE check_results ADD COLUMN llm_analysis TEXT")
            except:
                pass

        conn.commit()

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/db.py


In [6]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/models/endpoint.py

from dataclasses import dataclass

@dataclass
class Endpoint:
    id: int | None
    name: str
    url: str
    enabled: bool = True
    expected_latency_ms: int = 500


Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/models/endpoint.py


In [7]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/endpoint_repository.py

from models.endpoint import Endpoint
from repositories.db import get_conn


class EndpointRepository:

    def add(self, endpoint: Endpoint) -> Endpoint:
        with get_conn() as conn:
            cur = conn.execute(
                "INSERT INTO endpoints (name, url, enabled, expected_latency_ms) VALUES (?, ?, ?, ?)",
                (endpoint.name, endpoint.url, 1 if endpoint.enabled else 0, endpoint.expected_latency_ms)
            )
            endpoint.id = cur.lastrowid
            conn.commit()

        return endpoint

    def list_all(self):
        with get_conn() as conn:
            rows = conn.execute(
                "SELECT id, name, url, enabled, expected_latency_ms FROM endpoints"
            ).fetchall()

        return [
            Endpoint(
                id=row[0],
                name=row[1],
                url=row[2],
                enabled=bool(row[3]),
                expected_latency_ms=row[4]
            )
            for row in rows
        ]

    def delete(self, endpoint_id: int):
        with get_conn() as conn:
            conn.execute("DELETE FROM check_results WHERE endpoint_id = ?", (endpoint_id,))
            conn.execute("DELETE FROM endpoints WHERE id = ?", (endpoint_id,))
            conn.commit()

    def clear_all(self):
        with get_conn() as conn:
            conn.execute("DELETE FROM check_results")
            conn.execute("DELETE FROM endpoints")
            conn.commit()

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/endpoint_repository.py


In [8]:
!ls /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/database

app.db


In [9]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/models/check_result.py
from dataclasses import dataclass

@dataclass
class CheckResult:
    id: int | None
    endpoint_id: int
    ts: str
    latency_ms: int | None
    status: str
    http_status: int | None
    error_type: str | None
    error_message: str | None
    llm_label: str | None = None
    llm_analysis: str | None = None

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/models/check_result.py


In [10]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/check_result_repository.py

from models.check_result import CheckResult
from repositories.db import get_conn


class CheckResultRepository:

    def add(self, result: CheckResult) -> CheckResult:
        with get_conn() as conn:
            cur = conn.execute(
                """
                INSERT INTO check_results
                (endpoint_id, ts, latency_ms, status, http_status, error_type, error_message, llm_label, llm_analysis)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    result.endpoint_id,
                    result.ts,
                    result.latency_ms,
                    result.status,
                    result.http_status,
                    result.error_type,
                    result.error_message,
                    result.llm_label,
                    result.llm_analysis,
                )
            )
            result.id = cur.lastrowid
            conn.commit()
        return result

    def list_all(self):
        with get_conn() as conn:
            rows = conn.execute(
                """
                SELECT id, endpoint_id, ts, latency_ms, status, http_status, error_type, error_message, llm_label, llm_analysis
                FROM check_results
                ORDER BY id DESC
                """
            ).fetchall()

        return [
            CheckResult(
                id=row[0],
                endpoint_id=row[1],
                ts=row[2],
                latency_ms=row[3],
                status=row[4],
                http_status=row[5],
                error_type=row[6],
                error_message=row[7],
                llm_label=row[8],
                llm_analysis=row[9],
            )
            for row in rows
        ]

    def clear_all(self):
        with get_conn() as conn:
            conn.execute("DELETE FROM check_results")
            conn.commit()

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/repositories/check_result_repository.py


In [11]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/probe/http_checker.py

import time
from datetime import datetime
import requests

from models.check_result import CheckResult
from models.endpoint import Endpoint

session = requests.Session()

class HttpChecker:
    def check(self, endpoint: Endpoint) -> CheckResult:
        start = time.time()

        try:
            response = session.head(endpoint.url, timeout=5, allow_redirects=True)
            latency_ms = int((time.time() - start) * 1000)

            return CheckResult(
                id=None,
                endpoint_id=endpoint.id,
                ts=datetime.now().isoformat(),
                latency_ms=latency_ms,
                status="OK" if response.ok else "ERROR",
                http_status=response.status_code,
                error_type=None if response.ok else "HTTP_ERROR",
                error_message=None if response.ok else f"HTTP {response.status_code}",
            )

        except requests.exceptions.Timeout:
            return CheckResult(
                id=None,
                endpoint_id=endpoint.id,
                ts=datetime.now().isoformat(),
                latency_ms=None,
                status="ERROR",
                http_status=None,
                error_type="TIMEOUT",
                error_message="Request timed out",
            )

        except requests.exceptions.RequestException as e:
            return CheckResult(
                id=None,
                endpoint_id=endpoint.id,
                ts=datetime.now().isoformat(),
                latency_ms=None,
                status="ERROR",
                http_status=None,
                error_type="REQUEST_ERROR",
                error_message=str(e),
            )

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/probe/http_checker.py


In [12]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/main.py

from repositories.db import migrate
from repositories.endpoint_repository import EndpointRepository
from models.endpoint import Endpoint


def main():
    migrate()

    repo = EndpointRepository()

    # Optional: only add a default endpoint if the database is empty
    if not repo.list_all():
        repo.add(Endpoint(
            id=None,
            name="Google",
            url="https://google.com"
        ))

    print("NetGuard initialized successfully.")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/main.py


In [13]:
!pip install requests

In [14]:
!pip install streamlit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 22.2 MB/s eta 0:00:00


In [15]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/dashboard.py

import sqlite3
from pathlib import Path
import pandas as pd
import streamlit as st

DB_PATH = Path(__file__).resolve().parents[1] / "database" / "app.db"

st.set_page_config(page_title="Network Error Detection Dashboard", layout="wide")

st.title("Network Error Detection Dashboard")
st.write("Basic prototype UI for monitoring endpoints and viewing check results.")

# Safety check so app doesn't crash if DB doesn't exist
if not DB_PATH.exists():
    st.error(f"Database not found: {DB_PATH}")
    st.stop()

conn = sqlite3.connect(DB_PATH)

endpoints_df = pd.read_sql_query("SELECT * FROM endpoints", conn)
results_df = pd.read_sql_query("SELECT * FROM check_results ORDER BY id DESC", conn)

conn.close()

col1, col2 = st.columns(2)

with col1:
    st.subheader("Endpoints")
    st.dataframe(endpoints_df, use_container_width=True)

with col2:
    st.subheader("Check Results")
    st.dataframe(results_df, use_container_width=True)

st.subheader("Summary")

total_endpoints = len(endpoints_df)
total_results = len(results_df)
ok_results = len(results_df[results_df["status"] == "OK"]) if not results_df.empty else 0
error_results = len(results_df[results_df["status"] == "ERROR"]) if not results_df.empty else 0

c1, c2, c3, c4 = st.columns(4)
c1.metric("Endpoints", total_endpoints)
c2.metric("Total Checks", total_results)
c3.metric("OK Results", ok_results)
c4.metric("Errors", error_results)

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/dashboard.py


In [16]:
import sqlite3
import pandas as pd

db_path = "/content/drive/MyDrive/CapstoneNetGuard/network_error_detection/database/app.db"
conn = sqlite3.connect(db_path)

endpoints_df = pd.read_sql_query("SELECT * FROM endpoints", conn)
results_df = pd.read_sql_query("SELECT * FROM check_results ORDER BY id DESC", conn)

conn.close()

print("Endpoints")
display(endpoints_df)

print("Check Results")
display(results_df)

Endpoints


,id,name,url,enabled,expected_latency_ms
0,1,Google,https://google.com,1,500


Check Results


,id,endpoint_id,ts,latency_ms,status,http_status,error_type,error_message,llm_label,llm_analysis
0,8,1,2026-03-17T22:07:35.782531,129,OK,200,None,None,None,None
1,7,1,2026-03-17T19:30:23.158751,95,OK,200,None,None,None,None
2,6,1,2026-03-17T18:47:39.863058,116,OK,200,None,None,None,None
3,5,1,2026-03-17T18:46:43.059503,128,OK,200,None,None,None,None
4,4,1,2026-03-17T18:43:22.942142,75,OK,200,None,None,None,None
5,3,1,2026-03-17T18:39:48.400359,104,OK,200,None,None,None,None
6,2,1,2026-03-17T18:37:43.300007,117,OK,200,None,None,None,None
7,1,1,2026-03-17T18:27:40.005721,76,OK,200,None,None,None,None


In [17]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/analysis/llm_analyzer.py

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class LLMAnalyzer:
    def __init__(self):
        model_name = "google/flan-t5-small"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def _llm_label(self, endpoint_name, result):
        prompt = f"""
You are a network monitoring classifier.

Choose exactly one label from this list:
NORMAL
HIGH_LATENCY
OUTAGE
REQUEST_FAILURE

Monitoring result:
Endpoint: {endpoint_name}
Status: {result.status}
Latency: {result.latency_ms}
HTTP Status: {result.http_status}
Error Type: {result.error_type}
Error Message: {result.error_message}

Answer with only one label.
"""

        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip().upper()

    def analyze_result(self, endpoint_name, result):
        raw_label = self._llm_label(endpoint_name, result)

        # Guardrails so obviously wrong answers get corrected
        if result.status == "OK" and result.http_status == 200:
            if result.latency_ms is not None and result.latency_ms >= 300:
                label = "HIGH_LATENCY"
                explanation = "The endpoint is reachable, but response time is higher than expected."
            else:
                label = "NORMAL"
                explanation = "The endpoint appears to be operating normally."
        elif result.status == "ERROR":
            if result.error_type in ["TIMEOUT", "REQUEST_ERROR"]:
                label = "OUTAGE"
                explanation = "The endpoint appears unreachable and may be experiencing an outage."
            else:
                label = "REQUEST_FAILURE"
                explanation = "The endpoint returned an error and may be experiencing a request failure."
        else:
            # fallback to LLM label if state is unclear
            if "HIGH_LATENCY" in raw_label:
                label = "HIGH_LATENCY"
                explanation = "The endpoint is reachable, but response time is higher than expected."
            elif "OUTAGE" in raw_label:
                label = "OUTAGE"
                explanation = "The endpoint appears unreachable and may be experiencing an outage."
            elif "REQUEST_FAILURE" in raw_label:
                label = "REQUEST_FAILURE"
                explanation = "The endpoint returned an error and may be experiencing a request failure."
            else:
                label = "NORMAL"
                explanation = "The endpoint appears to be operating normally."

        return raw_label, label, explanation

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/analysis/llm_analyzer.py


In [18]:
%%writefile /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/gui.py

import sqlite3
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import pandas as pd

from repositories.db import migrate
from repositories.endpoint_repository import EndpointRepository
from repositories.check_result_repository import CheckResultRepository
from models.endpoint import Endpoint
from probe.http_checker import HttpChecker
from analysis.llm_analyzer import LLMAnalyzer

DB_PATH = Path(__file__).resolve().parents[1] / "database" / "app.db"


class NetGuardGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("NetGuard Network Error Detection Dashboard")
        self.root.geometry("1400x850")

        migrate()

        self.endpoint_repo = EndpointRepository()
        self.result_repo = CheckResultRepository()
        self.checker = HttpChecker()
        self.analyzer = LLMAnalyzer()

        title = tk.Label(
            root,
            text="NetGuard Network Error Detection Dashboard",
            font=("Arial", 20, "bold")
        )
        title.pack(pady=10)

        subtitle = tk.Label(
            root,
            text="Prototype interface for monitoring endpoints, running checks, and viewing LLM-assisted results",
            font=("Arial", 11)
        )
        subtitle.pack(pady=5)

        summary_frame = tk.Frame(root)
        summary_frame.pack(pady=10)

        self.endpoints_label = tk.Label(summary_frame, text="Endpoints: 0", font=("Arial", 11, "bold"))
        self.endpoints_label.grid(row=0, column=0, padx=20)

        self.checks_label = tk.Label(summary_frame, text="Total Checks: 0", font=("Arial", 11, "bold"))
        self.checks_label.grid(row=0, column=1, padx=20)

        self.ok_label = tk.Label(summary_frame, text="OK Results: 0", font=("Arial", 11, "bold"))
        self.ok_label.grid(row=0, column=2, padx=20)

        self.error_label = tk.Label(summary_frame, text="Errors: 0", font=("Arial", 11, "bold"))
        self.error_label.grid(row=0, column=3, padx=20)

        add_frame = tk.LabelFrame(root, text="Add Endpoint", font=("Arial", 11, "bold"))
        add_frame.pack(fill="x", padx=10, pady=10)

        tk.Label(add_frame, text="Name:", font=("Arial", 10)).grid(row=0, column=0, padx=10, pady=10, sticky="w")
        self.name_entry = tk.Entry(add_frame, width=30, font=("Arial", 10))
        self.name_entry.grid(row=0, column=1, padx=10, pady=10)

        tk.Label(add_frame, text="URL:", font=("Arial", 10)).grid(row=0, column=2, padx=10, pady=10, sticky="w")
        self.url_entry = tk.Entry(add_frame, width=50, font=("Arial", 10))
        self.url_entry.grid(row=0, column=3, padx=10, pady=10)

        add_button = tk.Button(
            add_frame,
            text="Add Endpoint",
            font=("Arial", 10, "bold"),
            command=self.add_endpoint
        )
        add_button.grid(row=0, column=4, padx=15, pady=10)

        content_frame = tk.Frame(root)
        content_frame.pack(fill="both", expand=True, padx=10, pady=10)

        left_frame = tk.LabelFrame(content_frame, text="Endpoints", font=("Arial", 11, "bold"))
        left_frame.pack(side="left", fill="both", expand=True, padx=5, pady=5)

        right_frame = tk.LabelFrame(content_frame, text="Check Results", font=("Arial", 11, "bold"))
        right_frame.pack(side="right", fill="both", expand=True, padx=5, pady=5)

        self.endpoint_tree = ttk.Treeview(
            left_frame,
            columns=("id", "name", "url", "enabled", "expected_latency"),
            show="headings",
            height=18
        )
        self.endpoint_tree.pack(fill="both", expand=True)

        self.endpoint_tree.heading("id", text="ID")
        self.endpoint_tree.heading("name", text="Name")
        self.endpoint_tree.heading("url", text="URL")
        self.endpoint_tree.heading("enabled", text="Enabled")
        self.endpoint_tree.heading("expected_latency", text="Expected Latency")

        self.endpoint_tree.column("id", width=50)
        self.endpoint_tree.column("name", width=140)
        self.endpoint_tree.column("url", width=300)
        self.endpoint_tree.column("enabled", width=80)
        self.endpoint_tree.column("expected_latency", width=140)

        self.result_tree = ttk.Treeview(
            right_frame,
            columns=("id", "endpoint_id", "ts", "latency", "status", "http_status", "llm_label"),
            show="headings",
            height=18
        )
        self.result_tree.pack(fill="both", expand=True)

        self.result_tree.heading("id", text="ID")
        self.result_tree.heading("endpoint_id", text="Endpoint ID")
        self.result_tree.heading("ts", text="Timestamp")
        self.result_tree.heading("latency", text="Latency (ms)")
        self.result_tree.heading("status", text="Status")
        self.result_tree.heading("http_status", text="HTTP Code")
        self.result_tree.heading("llm_label", text="LLM Label")

        self.result_tree.column("id", width=50)
        self.result_tree.column("endpoint_id", width=90)
        self.result_tree.column("ts", width=180)
        self.result_tree.column("latency", width=95)
        self.result_tree.column("status", width=80)
        self.result_tree.column("http_status", width=90)
        self.result_tree.column("llm_label", width=120)

        llm_frame = tk.LabelFrame(root, text="Latest LLM Analysis", font=("Arial", 11, "bold"))
        llm_frame.pack(fill="x", padx=10, pady=10)

        self.llm_text = tk.Text(llm_frame, height=5, wrap="word", font=("Arial", 10))
        self.llm_text.pack(fill="x", padx=10, pady=10)

        button_frame = tk.Frame(root)
        button_frame.pack(pady=10)

        check_button = tk.Button(
            button_frame,
            text="Check Selected Endpoint",
            font=("Arial", 11, "bold"),
            command=self.run_check_selected
        )
        check_button.grid(row=0, column=0, padx=10)

        delete_button = tk.Button(
            button_frame,
            text="Delete Selected Endpoint",
            font=("Arial", 11, "bold"),
            command=self.delete_selected_endpoint
        )
        delete_button.grid(row=0, column=1, padx=10)

        clear_button = tk.Button(
            button_frame,
            text="Clear All Data",
            font=("Arial", 11, "bold"),
            command=self.clear_all_data
        )
        clear_button.grid(row=0, column=2, padx=10)

        export_button = tk.Button(
            button_frame,
            text="Export Results to Excel",
            font=("Arial", 11, "bold"),
            command=self.export_to_excel
        )
        export_button.grid(row=0, column=3, padx=10)

        self.load_data()

    def get_conn(self):
        return sqlite3.connect(DB_PATH)

    def add_endpoint(self):
        name = self.name_entry.get().strip()
        url = self.url_entry.get().strip()

        if not name or not url:
            messagebox.showerror("Input Error", "Please enter both a name and a URL.")
            return

        if not url.startswith("http://") and not url.startswith("https://"):
            url = "https://" + url

        try:
            self.endpoint_repo.add(Endpoint(
                id=None,
                name=name,
                url=url
            ))

            self.name_entry.delete(0, tk.END)
            self.url_entry.delete(0, tk.END)

            self.load_data()
            messagebox.showinfo("Success", f"Endpoint '{name}' added successfully.")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to add endpoint:\n{e}")

    def run_check_selected(self):
        selected = self.endpoint_tree.selection()

        if not selected:
            messagebox.showwarning("No Selection", "Please select an endpoint first.")
            return

        item = self.endpoint_tree.item(selected[0])
        endpoint_id = item["values"][0]
        endpoint_name = item["values"][1]
        endpoint_url = item["values"][2]
        endpoint_enabled = bool(item["values"][3])
        expected_latency = item["values"][4]

        endpoint = Endpoint(
            id=endpoint_id,
            name=endpoint_name,
            url=endpoint_url,
            enabled=endpoint_enabled,
            expected_latency_ms=expected_latency
        )

        result = self.checker.check(endpoint)
        raw_label, final_label, final_explanation = self.analyzer.analyze_result(endpoint.name, result)

        result.llm_label = final_label
        result.llm_analysis = final_explanation

        self.result_repo.add(result)
        self.load_data()

        messagebox.showinfo("Check Complete", f"Finished checking '{endpoint.name}'.")

    def delete_selected_endpoint(self):
        selected = self.endpoint_tree.selection()

        if not selected:
            messagebox.showwarning("No Selection", "Please select an endpoint to delete.")
            return

        item = self.endpoint_tree.item(selected[0])
        endpoint_id = item["values"][0]
        endpoint_name = item["values"][1]

        confirm = messagebox.askyesno(
            "Confirm Delete",
            f"Delete endpoint '{endpoint_name}' and all of its check results?"
        )

        if confirm:
            self.endpoint_repo.delete(endpoint_id)
            self.load_data()

    def clear_all_data(self):
        confirm = messagebox.askyesno(
            "Confirm Clear",
            "This will delete ALL endpoints and ALL check results. Continue?"
        )

        if confirm:
            self.endpoint_repo.clear_all()
            self.load_data()

    def export_to_excel(self):
        try:
            conn = self.get_conn()

            query = """
            SELECT
                cr.id AS Check_ID,
                e.name AS Endpoint_Name,
                e.url AS Endpoint_URL,
                cr.ts AS Timestamp,
                cr.latency_ms AS Latency_ms,
                cr.status AS Status,
                cr.http_status AS HTTP_Code,
                cr.error_type AS Error_Type,
                cr.error_message AS Error_Message,
                cr.llm_label AS LLM_Label,
                cr.llm_analysis AS LLM_Analysis
            FROM check_results cr
            JOIN endpoints e ON cr.endpoint_id = e.id
            ORDER BY cr.id DESC
            """

            df = pd.read_sql_query(query, conn)
            conn.close()

            if df.empty:
                messagebox.showwarning("No Data", "There are no check results to export.")
                return

            file_path = filedialog.asksaveasfilename(
                defaultextension=".xlsx",
                filetypes=[("Excel files", "*.xlsx")],
                title="Save Excel Report",
                initialfile="netguard_check_results.xlsx"
            )

            if not file_path:
                return

            with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
                df.to_excel(writer, index=False, sheet_name="Check Results")

                worksheet = writer.sheets["Check Results"]

                for column_cells in worksheet.columns:
                    max_length = 0
                    column_letter = column_cells[0].column_letter
                    for cell in column_cells:
                        try:
                            if cell.value:
                                max_length = max(max_length, len(str(cell.value)))
                        except:
                            pass
                    worksheet.column_dimensions[column_letter].width = max_length + 2

            messagebox.showinfo("Export Complete", f"Excel file saved successfully:\n{file_path}")

        except Exception as e:
            messagebox.showerror("Export Error", f"Failed to export Excel file:\n{e}")

    def load_data(self):
        for item in self.endpoint_tree.get_children():
            self.endpoint_tree.delete(item)

        for item in self.result_tree.get_children():
            self.result_tree.delete(item)

        conn = self.get_conn()

        endpoints = conn.execute(
            "SELECT id, name, url, enabled, expected_latency_ms FROM endpoints"
        ).fetchall()

        results = conn.execute(
            """
            SELECT id, endpoint_id, ts, latency_ms, status, http_status, llm_label, llm_analysis
            FROM check_results
            ORDER BY id DESC
            """
        ).fetchall()

        conn.close()

        for row in endpoints:
            self.endpoint_tree.insert("", "end", values=(row[0], row[1], row[2], row[3], row[4]))

        for row in results:
            self.result_tree.insert("", "end", values=(row[0], row[1], row[2], row[3], row[4], row[5], row[6]))

        total_endpoints = len(endpoints)
        total_checks = len(results)
        ok_results = len([r for r in results if r[4] == "OK"])
        error_results = len([r for r in results if r[4] == "ERROR"])

        self.endpoints_label.config(text=f"Endpoints: {total_endpoints}")
        self.checks_label.config(text=f"Total Checks: {total_checks}")
        self.ok_label.config(text=f"OK Results: {ok_results}")
        self.error_label.config(text=f"Errors: {error_results}")

        self.llm_text.delete("1.0", tk.END)
        if results:
            latest_analysis = results[0][7]
            if latest_analysis:
                self.llm_text.insert(tk.END, latest_analysis)
            else:
                self.llm_text.insert(tk.END, "No LLM analysis available yet.")
        else:
            self.llm_text.insert(tk.END, "No check results available yet.")


if __name__ == "__main__":
    root = tk.Tk()
    app = NetGuardGUI(root)
    root.mainloop()

Overwriting /content/drive/MyDrive/CapstoneNetGuard/network_error_detection/app/gui.py
